In [1]:
import sys, time, platform, statistics
import pandas as pd

REPO = "/home/ibm-user/shared/safecircle"
sys.path.insert(0, REPO)
from src.messages import score_message
from src.transactions import score_transaction
from src.risk import decide
from src.explain import redact, explain

print("Running on:", platform.machine())

Running on: s390x


In [2]:
messages     = pd.read_csv(f"{REPO}/scenarios/messages.csv")
transactions = pd.read_csv(f"{REPO}/scenarios/transactions.csv")
messages.columns     = messages.columns.str.strip().str.lower()
transactions.columns = transactions.columns.str.strip().str.lower()

msg_text = messages.set_index("user_id")["text"]
TXN_KEYS = ["user_id", "txn_id", "timestamp", "amount", "merchant",
            "merchant_category", "is_new_payee"]

cases = [(msg_text.get(t["user_id"], ""), t[TXN_KEYS].to_dict())
         for _, t in transactions.iterrows()]
print(len(cases), "scenarios loaded:", [c[1]["user_id"] for c in cases])

3 scenarios loaded: ['rosa', 'walter', 'linda']


In [3]:
N = 1000
steps = {"message model": [], "payment model": [], "decide": [],
         "redact + explain": [], "total": []}

# Warm-up: the first call loads model files from disk, which isn't part of a real decision
text, txn = cases[0]
explain(redact(text), decide(score_message(text), score_transaction(txn)))

for i in range(N):
    text, txn = cases[i % len(cases)]
    t0 = time.perf_counter(); m = score_message(text) if text else {"score": 0.0, "reasons": []}
    t1 = time.perf_counter(); t = score_transaction(txn)
    t2 = time.perf_counter(); d = decide(m, t)
    t3 = time.perf_counter(); explain(redact(text), d)
    t4 = time.perf_counter()
    for name, a, b in [("message model", t0, t1), ("payment model", t1, t2),
                       ("decide", t2, t3), ("redact + explain", t3, t4), ("total", t0, t4)]:
        steps[name].append((b - a) * 1000)

speed = pd.DataFrame({
    name: {"median ms": statistics.median(v),
           "slowest 5% ms": sorted(v)[int(0.95 * N)],
           "share of total": statistics.median(v) / statistics.median(steps["total"])}
    for name, v in steps.items()}).T
speed["share of total"] = (speed["share of total"] * 100).round(0).astype(int).astype(str) + "%"
speed[["median ms", "slowest 5% ms"]] = speed[["median ms", "slowest 5% ms"]].round(3)

print(f"{N} payment decisions on {platform.machine()} (IBM LinuxONE)")
print(f"Decisions per second: {1000 / statistics.median(steps['total']):,.0f}")
speed

1000 payment decisions on s390x (IBM LinuxONE)
Decisions per second: 145


,median ms,slowest 5% ms,share of total
message model,0.820,0.919,12%
payment model,6.106,6.413,88%
decide,0.004,0.005,0%
redact + explain,0.040,0.053,1%
total,6.913,7.327,100%
